In [26]:
from jinja2 import Template
import pandas as pd
import os
import yaml
import sys
import json
import random
import hashlib
from itertools import product
from typing import Dict
import torch as t
from transformers import AutoTokenizer, AutoModelForCausalLM
from pydantic import BaseModel
import outlines
from outlines import Generator
from openai import OpenAI
from tqdm import tqdm
sys.path.append("../")
from src.utils import list_to_str, openai_api_call
device = "cpu"

In [28]:
class OutputFormat(BaseModel):
    score: float
    justification: str
    
class SJTLLMJudge(BaseModel):
    scenario_realism: OutputFormat
    trait_alignment: OutputFormat
    distinctiveness: OutputFormat
    ethical_tension: OutputFormat
    clarity: OutputFormat
    fairness: OutputFormat
    psychometric_value: OutputFormat

In [2]:
def write_to_json(file, file_path):
    with open(file_path, 'w') as f:
        json.dump(file, f)
        
def read_json(file_path):
    with open(file_path, "r") as f:
        file = json.load(f)
    return file

In [4]:
synthetic_sjts = read_json('sjt_data/synthetic_generated_sjt_list.json')

In [17]:
SJT_LLM_JUDGE_EVALUATION_TEMPLATE_STR = """You are an expert evaluator of situational judgment tests (SJTs). Your role is to assess the quality of each SJT scenario and its response options using a structured rubric. You must score each dimension on a 1–5 scale and provide a concise justification. Be objective, consistent, and fair. Always return results in JSON format.

Evaluate the following SJT using the rubric provided.

**Situational Judgment Test:**

Question: {{ question }}

Answer Options:

{{ answer_options }}

**Question Description**
* Every option in the question corresponds to a HEXACO Trait and they follow the following order Honesty-Humility, Emotionality, Extraversion, Agreeableness, Conscientiousness, Openness to Experience.
* Question and answers are created using specific seed values.
* SJT is created to simulate a professional law enforcement context.

**Seed Description**
* **Urgency Level:**  
   * Low: Situation allows ample time for decision-making with no immediate pressure.  
   * Medium: Requires timely attention but still allows some deliberation.  
   * High: Demands rapid response with little to no time for delay.  
* **Threat Level:**  
   * Low: Minimal risk to safety or order; situation is stable.  
   * Medium: Moderate potential risk requiring caution and situational awareness.  
   * High: Significant danger present, with immediate risk to safety or security.  
* **Ambiguity Level:**  
   * Clear: Situation and expectations are straightforward with little uncertainty.  
   * Moderate: Some uncertainty or incomplete information, requiring judgment.  
   * High: High uncertainty with unclear information or conflicting signals.  
* **Individuals Involved:**  
   * Simple: Few people engaged, interactions are straightforward.  
   * Moderate: Several people with varying roles or interests are present.  
   * Complex: Many individuals involved, with diverse and possibly conflicting needs.  
* **Authority Relationships:**  
   * Peer Level: Interactions with fellow officers, colleagues, or equal-ranking partners.  
   * Subordinate: Interactions with supervisors, training officers, or senior personnel.  
   * Authority: Interactions with civilians, suspects, witnesses, or those under your command.  
* **Situation Type:**  
   * Patrol Traffic Stop: Routine or situational encounters with drivers, often involving vehicle checks, traffic violations, or suspicious behavior.  
   * Crime Scene Investigation: Processing, securing, and documenting a scene after a crime has occurred, including evidence collection.  
   * Emergency Response: Immediate, time-sensitive incidents such as accidents, natural disasters, or active threats requiring rapid decisions.  
   * Administrative Reporting: Non-field tasks like writing reports, handling paperwork, or completing compliance records.  
   * Training Supervision: Scenarios involving mentoring, evaluating, or guiding subordinates during training.  
   * Inter-Agency Cooperation: Coordinated operations with other agencies (local, state, federal, or specialized units).  
   * Mental Health Crises: Encounters with individuals in psychological distress, requiring de-escalation and empathy.  
* **Time of Day:**  
   * Morning: Early hours, often involving routine checks or follow-up tasks.  
   * Afternoon: Midday period with typical public activity and moderate workload.  
   * Evening: Later hours with increased incidents related to social activity or nightlife.  
   * Night: Overnight period, often lower staffing but higher risk emergencies.  
* **Race:**  
   * White: Individual identifies as White or of European descent.  
   * Black or African American: Individual identifies as Black or African American.  
   * Hispanic/Latino: Individual identifies as Hispanic or Latino, of any race.  
   * Asian: Individual identifies as Asian, including East, South, or Southeast Asian backgrounds.  
   * Native American or Alaska Native: Individual identifies as Indigenous to North America.  
   * Pacific Islander: Individual identifies as Native Hawaiian or from other Pacific Islander groups.  
   * Other/Multiracial: Individual identifies as multiple races or ethnicities not captured in one category.  
   * Unknown: Race not identified or not disclosed.  
* **Gender:**  
   * Male: Individual identifies as male.  
   * Female: Individual identifies as female.  
   * Non-Binary: Individual identifies outside the male/female binary.  
   * Unknown: Gender not identified or not disclosed.  
* **Age:**  
   * Juvenile: Child or adolescent, generally under 18 years.  
   * Young Adult: Late teens through mid-20s.  
   * Adult: Standard adult range, typically 25–39.  
   * Middle-Aged: Individuals in their 40s to late 50s.  
   * Senior: Older adults, usually 60 years or above.  
   * Unknown: Age not identified or not disclosed.  
   
**Attribute Values used to create the SJT:**
* **Urgency Level:** {{urgency_level}}
* **Threat Level:** {{threat_level}}
* **Ambiguity Level:** {{ambiguity_level}}
* **Individuals Involved:** {{individuals_involved}}
* **Authority Relationships:** {{authority_relationships}}
* **Ethical Considerations:** {{ethical_considerations}}
* **Situation Type:** {{situation_type}}
* **Time of Day:** {{time_of_day}}
* **Subject Race:** {{race}}
* **Subject Gender:** {{gender}}
* **Subject Age:** {{age}}

**Rubric Dimensions (rate each 1–5):**
* **Scenario Realism & Plausibility:** Is the scenario realistic and consistent with policing practice?
* **Trait Alignment of Options:** Do the six options clearly map to their intended HEXACO traits?
* **Distinctiveness & Non-Redundancy of Options:** Are the options unique and not overlapping?
* **Ethical & Value Tension Representation:** Does the scenario involve meaningful ethical or professional trade-offs?
* **Clarity & Cognitive Load:** Is the scenario clear and understandable without confusion?
* **Bias & Fairness Check:** Are demographic or contextual factors presented neutrally (no stereotypes)?
* **Response Richness & Psychometric Value:** Do the options provide meaningful trait-driven differentiation?

**Output Format**
Provide the complete rubric evaluation as a JSON object with the following schema:

{
  "scenario_realism": {
    "score": X,
    "justification": "Concise reasoning here"
  },
  "trait_alignment": {
    "score": X,
    "justification": "Concise reasoning here"
  },
  "distinctiveness": {
    "score": X,
    "justification": "Concise reasoning here"
  },
  "ethical_tension": {
    "score": X,
    "justification": "Concise reasoning here"
  },
  "clarity": {
    "score": X,
    "justification": "Concise reasoning here"
  },
  "fairness": {
    "score": X,
    "justification": "Concise reasoning here"
  },
  "psychometric_value": {
    "score": X,
    "justification": "Concise reasoning here"
  }
}

Do not include any extra text, explanation, or formatting outside of the JSON object.
"""

In [18]:
SJT_LLM_JUDGE_EVALUATION_TEMPLATE = Template(SJT_LLM_JUDGE_EVALUATION_TEMPLATE_STR)

In [54]:
sjt_llm_judge_evaluation_result = []
for sjt in synthetic_sjts[:2]:
    config_dict = sjt['config'].copy()
    config_dict['question'] = sjt['question']
    config_dict['answer_options'] = list_to_str([f"{key} : {sjt[key]}" for key in sjt.keys() if "_option" in key])
    
    sjt_evaluation_prompt = SJT_LLM_JUDGE_EVALUATION_TEMPLATE.render(config_dict)
    openai_sjt_response = openai_api_call(prompt=sjt_evaluation_prompt, response_format=SJTLLMJudge ,model="gpt-4.1-mini")
    
    response = openai_sjt_response.model_dump()
    response['question_hash_id'] = sjt['hash_id']
    sjt_llm_judge_evaluation_result.append(response)

In [60]:
json.dumps(sjt_llm_judge_evaluation_result)

'[{"scenario_realism": {"score": 5.0, "justification": "The scenario depicts a realistic and plausible law enforcement situation involving a common welfare check with an agitated individual and conflicting supervisory instructions, occurring at night with high urgency and ambiguity."}, "trait_alignment": {"score": 5.0, "justification": "Each answer option clearly corresponds to a specific HEXACO trait, reflecting characteristic behaviors and decision approaches consistent with those traits."}, "distinctiveness": {"score": 5.0, "justification": "Options are well differentiated, presenting unique strategies and priorities that avoid redundancy, enabling clear distinctions among traits."}, "ethical_tension": {"score": 5.0, "justification": "The scenario adeptly captures meaningful ethical conflicts between transparency and self-protection, supervisor directives, and procedural rigor under ambiguous guidance."}, "clarity": {"score": 4.0, "justification": "The scenario and options are gener